In [9]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print("Q1 Answer:", len(documents))

Q1 Answer: 72


In [10]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

print("\nQ2 Results:")

for r in results:
    print(r["filename"])

print("\nQ2 Answer:", results[0]["filename"])


Q2 Results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md

Q2 Answer: 01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    base_url="https://ai-gateway.vercel.sh/v1",
    
    api_key=os.getenv("AI_GATEWAY_API_KEY"), 
)

In [12]:
def search(query):
    return index.search(
        query=query,
        num_results=5
    )

In [13]:
def build_context(results):

    context = ""

    for doc in results:
        context += f"""
FILE: {doc['filename']}

{doc['content']}

=========================
"""

    return context

In [15]:
def rag(query):

    results = search(query)

    context = build_context(results)

    prompt = f"""
You are a teaching assistant.

Answer the question based on the context.

QUESTION:
{query}

CONTEXT:
{context}
"""

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    answer = response.output_text

    tokens = response.usage.input_tokens

    return answer, tokens

In [16]:
answer, tokens = rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("\nQ3 Answer:")
print(answer)

print("\nQ3 Input Tokens:")
print(tokens)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'AI Gateway requires a valid credit card on file to service requests. Please visit https://vercel.com/d?to=%2F%5Bteam%5D%2F%7E%2Fai%3Fmodal%3Dadd-credit-card to add a card and unlock your free credits.', 'type': 'customer_verification_required'}}